# NVIDIA predecoder statistics

Run this notebook from its containing directory with **Restart Kernel and Run All Cells**. Dependencies: `numpy`, `pandas`, `scipy`, and a Jupyter Python kernel.

Read the original CSV values without modifying the input files. Each file contains independent run summaries, with baseline and predecoder results matched by run. Preserve source filenames and log paths for traceability.

**Metric interpretation:** The color-code CSV explicitly reports LER **per round** (Z basis, 5 rounds, 4,096 shots per run). The surface-code CSV reports `ler_avg_no_predecoder` and `ler_avg_after_predecoder`, but provides neither round counts nor an explicit per-round designation. Its speedup column explicitly denotes an X/Z average; the averaging and normalization of its LER require the original benchmark implementation or logs to confirm. No round conversion or block-accuracy reconstruction is applied to either file. Source log paths are recorded in the CSVs; their contents are not supplied here.


In [1]:
from pathlib import Path
from IPython.display import display
from importlib.metadata import version
import numpy as np
import pandas as pd
from scipy.stats import t

for package in ["numpy", "pandas", "scipy"]:
    print(f"{package}: {version(package)}")

DATA_DIR = Path("datasets/NVIDIA_predecoders")
SPECS = {
    "Surface / PyMatching": {
        "file": "bench_surface_fast_cpu_results.csv",
        "baseline": "ler_avg_no_predecoder",
        "after": "ler_avg_after_predecoder",
        "speedup": "pymatching_speedup_avg_xz",
        "ler_definition": "Reported average LER; round normalization unspecified",
    },
    "Color d5 / Chromobius": {
        "file": "bench_color_modelB_cpu_d5_results.csv",
        "baseline": "ler_no_predecoder_per_round",
        "after": "ler_after_predecoder_per_round",
        "speedup": "chromobius_speedup",
        "ler_definition": "LER per round (Z basis)",
    },
}

raw_results = {}
frames = []
for benchmark, spec in SPECS.items():
    path = DATA_DIR / spec["file"]
    raw = pd.read_csv(path)
    required = ["run", "source", spec["baseline"], spec["after"], spec["speedup"]]
    missing = set(required) - set(raw.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")
    if raw.empty or raw["run"].isna().any() or raw["run"].duplicated().any():
        raise ValueError(f"{path}: empty input or invalid/duplicate run identifiers")
    raw_results[benchmark] = raw.copy()
    print(f"\n{benchmark}: {path} ({len(raw)} runs)")
    display(raw)
    frame = raw.rename(columns={
        spec["baseline"]: "ler_baseline",
        spec["after"]: "ler_after",
        spec["speedup"]: "speedup",
    }).copy()
    for column in ["ler_baseline", "ler_after", "speedup"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    values = frame[["ler_baseline", "ler_after", "speedup"]].to_numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"{path}: non-finite metrics")
    if not frame[["ler_baseline", "ler_after"]].apply(lambda s: s.between(0, 1)).all().all():
        raise ValueError(f"{path}: LER outside [0, 1]")
    if not frame["speedup"].gt(0).all():
        raise ValueError(f"{path}: speedup must be positive")
    frame.insert(0, "benchmark", benchmark)
    frame["ler_definition"] = spec["ler_definition"]
    frame["source_csv"] = path.as_posix()
    frames.append(frame)

results = pd.concat(frames, ignore_index=True)


numpy: 1.26.4
pandas: 2.2.3
scipy: 1.16.0

Surface / PyMatching: datasets/NVIDIA_predecoders/bench_surface_fast_cpu_results.csv (6 runs)


,run,ler_avg_no_predecoder,ler_avg_after_predecoder,pymatching_speedup_avg_xz,source
0,1,0.002075,0.001831,2.327,bench_surface_fast_cpu_run1/logs/inference.con...
1,2,0.001709,0.001709,2.120,bench_surface_fast_cpu_run2/logs/inference.con...
2,3,0.001343,0.002563,2.190,bench_surface_fast_cpu_run3/logs/inference.con...
3,4,0.001465,0.000854,2.224,bench_surface_fast_cpu_run4/logs/inference.con...
4,5,0.001831,0.001465,2.293,bench_surface_fast_cpu_run5/logs/inference.con...
5,6,0.001099,0.001587,2.178,bench_surface_fast_cpu_run6/logs/inference.con...



Color d5 / Chromobius: datasets/NVIDIA_predecoders/bench_color_modelB_cpu_d5_results.csv (6 runs)


,run,basis,shots,rounds,ler_no_predecoder_per_round,ler_after_predecoder_per_round,chromobius_speedup,source
0,1,Z,4096,5,0.010449,0.006494,3.4722,bench_color_modelB_cpu_d5_run1/logs/inference....
1,2,Z,4096,5,0.009668,0.006836,3.4134,bench_color_modelB_cpu_d5_run2/logs/inference....
2,3,Z,4096,5,0.010107,0.006494,3.3473,bench_color_modelB_cpu_d5_run3/logs/inference....
3,4,Z,4096,5,0.008789,0.006152,3.4731,bench_color_modelB_cpu_d5_run4/logs/inference....
4,5,Z,4096,5,0.008691,0.006787,3.3318,bench_color_modelB_cpu_d5_run5/logs/inference....
5,6,Z,4096,5,0.008154,0.005664,3.4009,bench_color_modelB_cpu_d5_run6/logs/inference....


## Paired changes in LER

Within each run, define `ler_change = ler_after - ler_baseline`: a negative value indicates improvement. Relative reduction is $100(\mathrm{baseline}-\mathrm{after})/\mathrm{baseline}$; positive values indicate improvement. Summarize these paired changes across runs, preserving the baseline/predecoder pairing. Mean relative reduction is the **mean of run-level percentages**, not the percentage change between the two overall means. A zero baseline yields an undefined relative reduction (`NaN`), excluded only from the relative-reduction summary.

The 95% CI for the mean paired difference describes the evidence for the direction of the change. A CI containing zero does not establish an improvement or deterioration at the two-sided 5% level under the stated t-model assumptions. These are separate, unadjusted intervals, not a simultaneous confidence statement across all metrics.


### LER improvement factor

Report the run-level ratio

$$F_{\mathrm{LER}}=\frac{\mathrm{LER}_{\mathrm{NoPD}}}{\mathrm{LER}_{\mathrm{AfterPD}}}.$$

Values above 1 indicate error reduction, 1 indicates no change, and values below 1 indicate increased LER. The primary improvement summary is the arithmetic mean of the run-level ratios, with sample SD and a Student's t 95% CI across independent runs. This differs from the ratio of the two overall mean LERs. A zero denominator produces an undefined or infinite ratio; in that case the notebook raises an error rather than silently excluding runs. The no-change reference for this factor is 1.


In [2]:
def summarize(values):
    values = np.asarray(values, dtype=float)
    count = len(values)
    if count == 0 or not np.isfinite(values).all():
        raise ValueError("Expected nonempty finite run-level values")
    mean = values.mean()
    sd = values.std(ddof=1) if count > 1 else np.nan
    margin = t.ppf(0.975, count - 1) * sd / np.sqrt(count) if count > 1 else np.nan
    return {"runs": count, "mean": mean, "std": sd,
            "ci95_low": mean - margin, "ci95_high": mean + margin}

summary = pd.DataFrame([
    {"benchmark": benchmark, "metric": metric, **summarize(group[metric])}
    for benchmark, group in results.groupby("benchmark", sort=False)
    for metric in ["ler_baseline", "ler_after", "speedup"]
])

if results["ler_after"].eq(0).any():
    raise ValueError("LER improvement factor cannot be summarized with zero post-predecoder LER.")
results["ler_improvement_factor"] = results["ler_baseline"] / results["ler_after"]
results["ler_change"] = results["ler_after"] - results["ler_baseline"]
results["relative_ler_reduction_pct"] = (
    -100 * results["ler_change"] / results["ler_baseline"].replace(0, np.nan)
)
paired_rows = []
for benchmark, group in results.groupby("benchmark", sort=False):
    for metric in ["ler_improvement_factor", "ler_change", "relative_ler_reduction_pct"]:
        values = group[metric].dropna()
        if len(values):
            paired_rows.append({"benchmark": benchmark, "metric": metric, **summarize(values)})
paired_summary = pd.DataFrame(paired_rows)
with pd.option_context("display.float_format", "{:.8f}".format):
    display(results[["benchmark", "run", "ler_baseline", "ler_after", "ler_change",
                     "ler_improvement_factor", "relative_ler_reduction_pct", "speedup"]])
    display(paired_summary)

for benchmark in SPECS:
    row = paired_summary.loc[
        (paired_summary["benchmark"] == benchmark) & (paired_summary["metric"] == "ler_change")
    ].iloc[0]
    if row["ci95_high"] < 0:
        conclusion = "lower LER after predecoding (paired 95% CI excludes zero)"
    elif row["ci95_low"] > 0:
        conclusion = "higher LER after predecoding (paired 95% CI excludes zero)"
    else:
        conclusion = "no clear directional LER change (paired 95% CI includes zero)"
    speed = summary.loc[(summary["benchmark"] == benchmark) & (summary["metric"] == "speedup")].iloc[0]
    print(f"{benchmark}: {conclusion}.")
    print(f"  Mean reported speedup: {speed['mean']:.4f}× "
          f"(95% CI [{speed['ci95_low']:.4f}, {speed['ci95_high']:.4f}]).")

# Primary LER improvement summary: paired per-run ratios, reference value = 1.
improvement_summary = paired_summary.loc[
    paired_summary["metric"].eq("ler_improvement_factor")
].reset_index(drop=True)
print("LER improvement factor (baseline / after predecoding):")
with pd.option_context("display.float_format", "{:.8f}".format):
    display(improvement_summary)


,benchmark,run,ler_baseline,ler_after,ler_change,ler_improvement_factor,relative_ler_reduction_pct,speedup
0,Surface / PyMatching,1,0.00207500,0.00183100,-0.00024400,1.13326051,11.75903614,2.32700000
1,Surface / PyMatching,2,0.00170900,0.00170900,0.00000000,1.00000000,-0.00000000,2.12000000
2,Surface / PyMatching,3,0.00134300,0.00256300,0.00122000,0.52399532,-90.84139985,2.19000000
3,Surface / PyMatching,4,0.00146500,0.00085400,-0.00061100,1.71545667,41.70648464,2.22400000
4,Surface / PyMatching,5,0.00183100,0.00146500,-0.00036600,1.24982935,19.98907701,2.29300000
5,Surface / PyMatching,6,0.00109900,0.00158700,0.00048800,0.69250158,-44.40400364,2.17800000
6,Color d5 / Chromobius,1,0.01044922,0.00649414,-0.00395508,1.60902256,37.85046729,3.47220000
7,Color d5 / Chromobius,2,0.00966797,0.00683594,-0.00283203,1.41428571,29.29292929,3.41340000
8,Color d5 / Chromobius,3,0.01010742,0.00649414,-0.00361328,1.55639098,35.74879227,3.34730000
9,Color d5 / Chromobius,4,0.00878906,0.00615234,-0.00263672,1.42857143,30.00000000,3.47310000


,benchmark,metric,runs,mean,std,ci95_low,ci95_high
0,Surface / PyMatching,ler_improvement_factor,6,1.05250724,0.42347255,0.60810005,1.49691442
1,Surface / PyMatching,ler_change,6,0.00008117,0.00067139,-0.00062341,0.00078575
2,Surface / PyMatching,relative_ler_reduction_pct,6,-10.29846762,48.71302464,-61.41965180,40.82271657
3,Color d5 / Chromobius,ler_improvement_factor,6,1.45475023,0.11577408,1.33325279,1.57624767
4,Color d5 / Chromobius,ler_change,6,-0.00290527,0.00075565,-0.00369828,-0.00211226
5,Color d5 / Chromobius,relative_ler_reduction_pct,6,30.89020389,5.58856582,25.02536375,36.75504404


Surface / PyMatching: no clear directional LER change (paired 95% CI includes zero).
  Mean reported speedup: 2.2220× (95% CI [2.1415, 2.3025]).
Color d5 / Chromobius: lower LER after predecoding (paired 95% CI excludes zero).
  Mean reported speedup: 3.4064× (95% CI [3.3436, 3.4693]).
LER improvement factor (baseline / after predecoding):


,benchmark,metric,runs,mean,std,ci95_low,ci95_high
0,Surface / PyMatching,ler_improvement_factor,6,1.05250724,0.42347255,0.60810005,1.49691442
1,Color d5 / Chromobius,ler_improvement_factor,6,1.45475023,0.11577408,1.33325279,1.57624767


## Surface-code results by run

LER improvement factor = No PD / After PD; values above 1 indicate improvement. The LER values and X/Z-average speedup are preserved as reported in the original surface-code CSV.

The summary below reports the mean and two-sided 95% Student's t interval across the six independent runs: mean ± t(0.975, 5) × sample SD / √6. These intervals assume approximately normally distributed run-level estimates. The improvement-factor interval uses individual run ratios; its no-change reference is 1. These are separate intervals for run-level means, not per-run intervals or pooled-shot binomial intervals.

In [3]:
surface_table = (
    results.loc[results["benchmark"].eq("Surface / PyMatching"),
                ["run", "ler_baseline", "ler_after", "ler_improvement_factor", "speedup"]]
    .sort_values("run")
    .rename(columns={
        "run": "Run",
        "ler_baseline": "No PD (Avg LER)",
        "ler_after": "After PD (Avg LER)",
        "ler_improvement_factor": "LER Improvement Factor",
        "speedup": "PyMatching Speedup (Avg X/Z)",
    })
    .reset_index(drop=True)
)
display(surface_table.style.hide(axis="index").format({
    "No PD (Avg LER)": "{:.6f}",
    "After PD (Avg LER)": "{:.6f}",
    "LER Improvement Factor": "{:.4f}×",
    "PyMatching Speedup (Avg X/Z)": "{:.4f}×",
}))

# Mean and two-sided 95% Student's t confidence interval across runs.
surface_ci95 = pd.DataFrame([
    {"Metric": column, **summarize(surface_table[column])}
    for column in surface_table.columns if column != "Run"
]).rename(columns={
    "runs": "Runs", "mean": "Mean", "std": "Sample SD",
    "ci95_low": "95% CI Lower", "ci95_high": "95% CI Upper",
})
print("Run-level means and 95% confidence intervals:")
display(surface_ci95.style.hide(axis="index").format({
    "Mean": "{:.8f}", "Sample SD": "{:.8f}",
    "95% CI Lower": "{:.8f}", "95% CI Upper": "{:.8f}",
}))


Run,No PD (Avg LER),After PD (Avg LER),LER Improvement Factor,PyMatching Speedup (Avg X/Z)
1,0.002075,0.001831,1.1333×,2.3270×
2,0.001709,0.001709,1.0000×,2.1200×
3,0.001343,0.002563,0.5240×,2.1900×
4,0.001465,0.000854,1.7155×,2.2240×
5,0.001831,0.001465,1.2498×,2.2930×
6,0.001099,0.001587,0.6925×,2.1780×


Run-level means and 95% confidence intervals:


Metric,Runs,Mean,Sample SD,95% CI Lower,95% CI Upper
No PD (Avg LER),6,0.00158700,0.00035359,0.00121593,0.00195807
After PD (Avg LER),6,0.00166817,0.00055491,0.00108582,0.00225051
LER Improvement Factor,6,1.05250724,0.42347255,0.60810005,1.49691442
PyMatching Speedup (Avg X/Z),6,2.22200000,0.07672548,2.14148154,2.30251846


## Color-code results by run

Both LER columns contain the original per-round LER values (Z basis). LER improvement factor = No PD / After PD. “Total Chromobius Speedup” displays the CSV `chromobius_speedup` values; the CSV alone does not establish which timing stages are included.

The summary below reports the mean and two-sided 95% Student's t interval across the six independent runs: mean ± t(0.975, 5) × sample SD / √6. These intervals assume approximately normally distributed run-level estimates. The improvement-factor interval uses individual run ratios; its no-change reference is 1. These are separate intervals for run-level means, not per-run intervals or pooled-shot binomial intervals.

In [4]:
color_table = (
    results.loc[results["benchmark"].eq("Color d5 / Chromobius"),
                ["run", "ler_baseline", "ler_after", "ler_improvement_factor", "speedup"]]
    .sort_values("run")
    .rename(columns={
        "run": "Run",
        "ler_baseline": "No PD (Avg LER)",
        "ler_after": "After PD (Avg LER)",
        "ler_improvement_factor": "LER Improvement Factor",
        "speedup": "Total Chromobius Speedup",
    })
    .reset_index(drop=True)
)
display(color_table.style.hide(axis="index").format({
    "No PD (Avg LER)": "{:.8f}",
    "After PD (Avg LER)": "{:.8f}",
    "LER Improvement Factor": "{:.4f}×",
    "Total Chromobius Speedup": "{:.4f}×",
}))

# Mean and two-sided 95% Student's t confidence interval across runs.
color_ci95 = pd.DataFrame([
    {"Metric": column, **summarize(color_table[column])}
    for column in color_table.columns if column != "Run"
]).rename(columns={
    "runs": "Runs", "mean": "Mean", "std": "Sample SD",
    "ci95_low": "95% CI Lower", "ci95_high": "95% CI Upper",
})
print("Run-level means and 95% confidence intervals:")
display(color_ci95.style.hide(axis="index").format({
    "Mean": "{:.8f}", "Sample SD": "{:.8f}",
    "95% CI Lower": "{:.8f}", "95% CI Upper": "{:.8f}",
}))


Run,No PD (Avg LER),After PD (Avg LER),LER Improvement Factor,Total Chromobius Speedup
1,0.01044922,0.00649414,1.6090×,3.4722×
2,0.00966797,0.00683594,1.4143×,3.4134×
3,0.01010742,0.00649414,1.5564×,3.3473×
4,0.00878906,0.00615234,1.4286×,3.4731×
5,0.00869141,0.00678711,1.2806×,3.3318×
6,0.00815430,0.00566406,1.4397×,3.4009×


Run-level means and 95% confidence intervals:


Metric,Runs,Mean,Sample SD,95% CI Lower,95% CI Upper
No PD (Avg LER),6,0.00930990,0.00090017,0.00836523,0.01025457
After PD (Avg LER),6,0.00640462,0.00043828,0.00594468,0.00686456
LER Improvement Factor,6,1.45475023,0.11577408,1.33325279,1.57624767
Total Chromobius Speedup,6,3.40645000,0.05985939,3.34363142,3.46926858
